# Pilot Arm G (ViT5-base) — Fair Comparison Benchmark
### Pretrained Vietnamese Sequence-to-Sequence vs. Custom Scratch Transformers (Arms A–F)

Notebook này thực hiện huấn luyện và đánh giá thực nghiệm độc lập cho **Arm G (VietAI/vit5-base)** theo đúng giao thức **Đối sánh sòng phẳng (Fair Comparison)**:
- **Cùng Dataset**: Đúng 300.000 cặp sạch `data/base_v3_pilot/train.src` và `train.tgt`.
- **Cùng Validation**: Đúng 5.000 cặp sạch `data/base_v3_pilot/valid.src` và `valid.tgt`.
- **Cùng Training Budget**: Đúng 10.000 steps.
- **Cùng Checkpoint đánh giá**: `step 10,000`.
- **Cùng Test Suites**: 5 Frozen Suites (3.150 truy vấn) + 18 ca lỗi thực tế `zero_click.csv`.
- **Cùng Metrics**: Exact-match accuracy, CPU INT8 latency (Beam 1 & Beam 4).

| Arm | Kiến trúc | Enc / Dec | Tham số | Khởi tạo | Động cơ kiểm chứng |
| :---: | :--- | :---: | :---: | :---: | :--- |
| **Arm A** | 2E1D d128 | 2E / 1D | 6.47M | Scratch | Baseline Production siêu tốc (2.38 ms) |
| **Arm E** | 2E2D d128 | 2E / 2D | 7.13M | Scratch | Top 1 Scaling Scratch (67.33% Acc, 5.71 ms) |
| **Arm B** | 4E4D d128 | 4E / 4D | 9.63M | Scratch | Top 2 Balanced Deep (67.11% Acc, 5.51 ms) |
| **Arm G** | **ViT5-base** | **12E / 12D** | **226M** | **Pretrained** | **Lợi thế thực sự của Pretraining tiếng Việt?** |


In [ ]:
# Build the complete runtime from the offline Kaggle wheelhouse.
# Keep Kaggle's CUDA-enabled torch; install every other Arm G dependency locally.
import os, sys, time, json, shutil, subprocess, math, re
from pathlib import Path

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
WHEELS_DIR = Path('/kaggle/input/datasets/vanhieu1125/build-wheel/wheels')
assert WHEELS_DIR.is_dir(), f'Missing offline wheelhouse: {WHEELS_DIR}'

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

wheel_names = [path.name.lower().replace('-', '_') for path in WHEELS_DIR.glob('*.whl')]
required_wheels = (
    'transformers_4.48.3', 'tokenizers_0.21.0', 'huggingface_hub_0.28.1',
    'safetensors_0.5.2', 'accelerate_', 'sentencepiece_0.2.0', 'ctranslate2_4.6.2',
)
missing_wheels = [prefix for prefix in required_wheels if not any(name.startswith(prefix) for name in wheel_names)]
assert not missing_wheels, f'Missing offline wheels in {WHEELS_DIR}: {missing_wheels}'
print(f'[+] Offline wheel preflight passed: {len(wheel_names)} wheels available')

# Install into an isolated directory. Kaggle may preload Transformers 5.0.0,
# so a normal pip reinstall does not reliably replace the module already on sys.path.
RUNTIME_DIR = Path('/kaggle/working/arm-g-site-packages')
if RUNTIME_DIR.exists():
    shutil.rmtree(RUNTIME_DIR)
RUNTIME_DIR.mkdir(parents=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-index', '--find-links', str(WHEELS_DIR),
    '--no-deps', '--target', str(RUNTIME_DIR),
    'transformers==4.48.3',
    'tokenizers==0.21.0',
    'huggingface_hub==0.28.1',
    'safetensors==0.5.2',
    'accelerate',
    'sentencepiece==0.2.0',
    'ctranslate2==4.6.2',
], check=True)

# Force imports to come from the isolated runtime, even if Kaggle preloaded v5 modules.
import importlib
sys.path.insert(0, str(RUNTIME_DIR))
# Only clear pure-Python packages. Never unload/reload native extensions such as
# ctranslate2, tokenizers, sentencepiece, or safetensors in one kernel.
for module_name in list(sys.modules):
    if module_name.split('.', 1)[0] in {
        'transformers', 'huggingface_hub', 'accelerate'
    }:
        del sys.modules[module_name]
importlib.invalidate_caches()

import torch, transformers, tokenizers, accelerate, sentencepiece, ctranslate2
assert transformers.__version__ == '4.48.3', transformers.__version__
assert tokenizers.__version__ == '0.21.0', tokenizers.__version__
assert sentencepiece.__version__ == '0.2.0', sentencepiece.__version__
assert ctranslate2.__version__ == '4.6.2', ctranslate2.__version__
print(f'Offline wheelhouse: {WHEELS_DIR}')
print(f'torch={torch.__version__} | transformers={transformers.__version__} | tokenizers={tokenizers.__version__}')
print(f'accelerate={accelerate.__version__} | sentencepiece={sentencepiece.__version__} | ctranslate2={ctranslate2.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'CUDA GPU is required for this Arm G run.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


In [ ]:
KAGGLE_INPUT = Path('/kaggle/input/datasets/vanhieu1125/test-bundle/data')
assert KAGGLE_INPUT.is_dir(), f'Missing training data root: {KAGGLE_INPUT}'
WORKING = Path('/kaggle/working')
REPO_ROOT = WORKING / "QU-solution"

# 1. Locate pilot dataset
pilot_dir = None
for p in KAGGLE_INPUT.rglob('base_v3_pilot'):
    if (p / 'train.src').exists():
        pilot_dir = p
        break

assert pilot_dir and (pilot_dir / 'train.src').exists(), f"[-] Could not find base_v3_pilot in {KAGGLE_INPUT}!"
for required in ('train.src', 'train.tgt', 'valid.src', 'valid.tgt'):
    assert (pilot_dir / required).is_file(), f'Missing pilot file: {pilot_dir / required}'
print(f"[+] Pilot training dataset located: {pilot_dir}")

# 2. Locate evaluation dataset
eval_dir = None
for p in [pilot_dir.parent / 'base_v3_eval', REPO_ROOT / 'data/base_v3_eval']:
    if p.exists() and (p / 'plasticity.src').exists():
        eval_dir = p
        break

if not eval_dir:
    for p in KAGGLE_INPUT.rglob('base_v3_eval'):
        if (p / 'plasticity.src').exists():
            eval_dir = p
            break

assert eval_dir and (eval_dir / 'plasticity.src').exists(), f"[-] Could not find base_v3_eval in {KAGGLE_INPUT}!"
print(f"[+] Frozen evaluation suites located: {eval_dir}")


In [ ]:
from torch.utils.data import Dataset
from transformers import T5Tokenizer

MODEL_ID = "VietAI/vit5-base"
# ViT5 is mounted as a separate Kaggle Dataset.
# A valid directory contains config.json, spiece.model and model weights.
def is_vit5_snapshot(path):
    has_weights = any((path / name).exists() for name in (
        'model.safetensors', 'pytorch_model.bin', 'model.safetensors.index.json',
        'pytorch_model.bin.index.json'))
    return (path / 'config.json').exists() and (path / 'spiece.model').exists() and has_weights

MODEL_PATH = Path('/kaggle/input/datasets/vanhieu1125/vit5-b/vit5-base')
assert is_vit5_snapshot(MODEL_PATH), (
    f'Invalid or incomplete offline ViT5 snapshot: {MODEL_PATH}. '
    'Expected config.json, spiece.model, and model weights.'
)
print(f'Loading offline tokenizer for {MODEL_ID}: {MODEL_PATH}')
# Force slow SentencePiece to avoid the native conversion KeyError seen on Kaggle.
tokenizer = T5Tokenizer.from_pretrained(
    str(MODEL_PATH), legacy=True, local_files_only=True
)
print(f'[+] Tokenizer backend: {type(tokenizer).__name__}')

class QueryCorrectionDataset(Dataset):
    def __init__(self, src_file, tgt_file, tokenizer, max_length=64):
        with open(src_file, 'r', encoding='utf-8') as f:
            src_lines = [line.rstrip('\r\n') for line in f]
        with open(tgt_file, 'r', encoding='utf-8') as f:
            tgt_lines = [line.rstrip('\r\n') for line in f]
        assert len(src_lines) == len(tgt_lines), f"Length mismatch: {len(src_lines)} vs {len(tgt_lines)}"
        pairs = [(src.strip().lower(), tgt.strip().lower()) for src, tgt in zip(src_lines, tgt_lines)]
        assert all(src and tgt for src, tgt in pairs), 'Blank source/target row found; refusing to shift pair alignment.'
        self.src_lines = [src for src, _ in pairs]
        self.tgt_lines = [tgt for _, tgt in pairs]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.src_lines)

    def __getitem__(self, idx):
        src_text = self.src_lines[idx]
        tgt_text = self.tgt_lines[idx]

        model_inputs = self.tokenizer(
            src_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        labels = self.tokenizer(
            tgt_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )["input_ids"].squeeze(0)

        # Mask padding in labels with -100 for CrossEntropyLoss
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": model_inputs["input_ids"].squeeze(0),
            "attention_mask": model_inputs["attention_mask"].squeeze(0),
            "labels": labels
        }

print("Loading train and valid datasets into memory...")
train_dataset = QueryCorrectionDataset(pilot_dir / 'train.src', pilot_dir / 'train.tgt', tokenizer, max_length=64)
valid_dataset = QueryCorrectionDataset(pilot_dir / 'valid.src', pilot_dir / 'valid.tgt', tokenizer, max_length=64)
assert len(train_dataset) == 300_000, f'Expected 300,000 train pairs, found {len(train_dataset):,}'
assert len(valid_dataset) == 5_000, f'Expected 5,000 validation pairs, found {len(valid_dataset):,}'
print(f"[+] Train samples: {len(train_dataset):,}")
print(f"[+] Valid samples: {len(valid_dataset):,}")


In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

OUTPUT_ROOT = WORKING / "checkpoints/arm_g_vit5_base"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Initializing {MODEL_ID}...")
model = AutoModelForSeq2SeqLM.from_pretrained(str(MODEL_PATH), local_files_only=True)
total_params = sum(p.numel() for p in model.parameters())
print(f"[+] {MODEL_ID} loaded successfully! Total parameters: {total_params / 1e6:.2f}M")

# Hyperparameters for exactly 10,000 steps
eval_kwarg = "eval_strategy" if hasattr(Seq2SeqTrainingArguments, "eval_strategy") else "evaluation_strategy"
training_kwargs = {
    eval_kwarg: "steps",
    "output_dir": str(OUTPUT_ROOT),
    "max_steps": 10_000,                  # Exactly 10,000 steps matching Arms A-F!
    "per_device_train_batch_size": 64,
    "per_device_eval_batch_size": 128,
    "gradient_accumulation_steps": 1,     # Effective batch size = 64 pairs; one micro-batch on RTX PRO 6000
    "learning_rate": 5e-5,               # Standard fine-tuning rate for T5/ViT5
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "logging_steps": 200,
    "save_steps": 5_000,                 # Checkpoints at 5k and 10k
    "save_total_limit": 2,
    "eval_steps": 2_500,
    "bf16": torch.cuda.is_available(),
    "fp16": False,
    # Disabled here: Transformers 4.48.3 mis-detects Blackwell compute capability 12.0.
    "tf32": False,
    "dataloader_num_workers": 4,
    "report_to": "none",
    "seed": 2026,
    "predict_with_generate": False,
}

training_args = Seq2SeqTrainingArguments(**training_kwargs)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
)

print("\n" + "=" * 80)
print("STARTING 10,000-STEP FINE-TUNING FOR PILOT ARM G (ViT5-base)")
print("=" * 80)
t0 = time.time()
trainer.train()
dur = (time.time() - t0) / 60.0
print(f"\n[+] Training finished in {dur:.1f} minutes!")

# Explicitly save final 10,000-step checkpoint
step_10k_dir = OUTPUT_ROOT / "checkpoint-10000"
trainer.save_model(str(step_10k_dir))
tokenizer.save_pretrained(str(step_10k_dir))
print(f"[+] Successfully saved step 10,000 model to {step_10k_dir}")


In [ ]:
CT2_DIR = OUTPUT_ROOT / "ct2_model"
if CT2_DIR.exists():
    shutil.rmtree(CT2_DIR)

print("\n" + "=" * 80)
print("CONVERTING ViT5-BASE CHECKPOINT (STEP 10,000) TO CTRANSLATE2 (FP16)")
print("=" * 80)

import ctranslate2.converters
converter = ctranslate2.converters.TransformersConverter(str(step_10k_dir))
converter.convert(str(CT2_DIR), quantization="float16", force=True)
assert (CT2_DIR / 'model.bin').is_file(), f'CTranslate2 conversion failed: {CT2_DIR}'
print(f"[+] CTranslate2 conversion successful: {CT2_DIR}")


In [ ]:
import ctranslate2

print("\n" + "=" * 80)
print("EVALUATING ARM G ON 5 FROZEN SUITES & REAL-WORLD ZERO-CLICK LOGS")
print("=" * 80)

SUITES = {
    "plasticity": (eval_dir / "plasticity.src", eval_dir / "plasticity.tgt"),
    "retention": (eval_dir / "retention.src", eval_dir / "retention.tgt"),
    "user_centric": (eval_dir / "user_centric.src", eval_dir / "user_centric.tgt"),
    "protection_seen": (eval_dir / "protection_seen.src", eval_dir / "protection_seen.tgt"),
    "protection_heldout": (eval_dir / "protection_heldout.src", eval_dir / "protection_heldout.tgt"),
}

for suite_name, (src_path, tgt_path) in SUITES.items():
    assert src_path.is_file(), f'Missing suite source: {src_path}'
    assert tgt_path.is_file(), f'Missing suite target: {tgt_path}'
    src_count = sum(1 for line in src_path.open(encoding='utf-8') if line.strip())
    tgt_count = sum(1 for line in tgt_path.open(encoding='utf-8') if line.strip())
    assert src_count == tgt_count, f'{suite_name}: {src_count} sources != {tgt_count} targets'
assert sum(sum(1 for line in src.open(encoding='utf-8') if line.strip()) for src, _ in SUITES.values()) == 3_100, 'Frozen suites must contain exactly 3,100 queries.'

ZERO_CLICK_CASES = [
    ("cau vuot song than",                         "cầu vượt sóng thần"),
    ("cho ba chieu",                               "chợ bà chiểu"),
    ("nga 4 hang xanh",                            "ngã 4 hàng xanh"),
    ("nga 3 vung tau",                             "ngã 3 vũng tàu"),
    ("bv cho ray",                                 "bệnh viện chợ rẫy"),
    ("Bv nhi đong",                                "bệnh viện nhi đồng"),
    ("benh vien 175",                              "bệnh viện 175"),
    ("dh kinh te tphcm",                           "đại học kinh tế thành phố hồ chí minh"),
    ("86 xo viet nghe tinh p19 binh thanh",        "86 xô viết nghệ tĩnh phường 19 bình thạnh"),
    ("duong le van viet q9",                       "đường lê văn việt quận 9"),
    ("hem 212 thoai ngoc hau phuong phu thanh",    "hẻm 212 thoại ngọc hầu phường phú thạnh"),
    ("kcn song than 1",                            "khu công nghiệp sóng thần 1"),
    ("ubnd xa phuoc thai",                         "ủy ban nhân dân xã phước thái"),
    ("158/16 binh quew",                           "158/16 bình quới"),
    ("chung cu ha",                                "chung cư hà"),
    ("ngã 6 tahnhf",                               "ngã 6 thành"),
    ("tttm aeon mall tan phu",                     "trung tâm thương mại aeon mall tân phú"),
    ("dh bach khoa ha noi",                        "đại học bách khoa hà nội"),
]

# Initialize CTranslate2 CPU Translator with INT8 (matching Arms A-F production benchmark)
translator_cpu = ctranslate2.Translator(str(CT2_DIR), device="cpu", compute_type="int8", intra_threads=4)

# Warmup
warm_toks = tokenizer.tokenize("ha noi")
translator_cpu.translate_batch([warm_toks], beam_size=1)

def predict_ct2(texts, beam_size=1, rep_penalty=1.2, batch_size=64):
    preds = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        token_batch = [tokenizer.tokenize(t.strip().lower()) for t in chunk]
        outputs = translator_cpu.translate_batch(token_batch, beam_size=beam_size, repetition_penalty=rep_penalty)
        for out in outputs:
            hyp = out.hypotheses[0]
            text = tokenizer.decode(tokenizer.convert_tokens_to_ids(hyp), skip_special_tokens=True)
            preds.append(text.strip().lower())
    return preds

def compute_acc(preds, tgts):
    assert len(preds) == len(tgts)
    c = sum(1 for p, t in zip(preds, tgts) if p.strip().lower() == t.strip().lower())
    return (c / len(preds)) * 100.0

arm_g_results = {
    "name": "Arm G (ViT5-base Pretrained)",
    "params": f"{total_params / 1e6:.1f}M",
    "suites": {},
    "latency": {},
    "zero_click_preds": {}
}

# Evaluate 5 suites
suite_accs = []
for s_name, (src_p, tgt_p) in SUITES.items():
    srcs = [l.strip() for l in open(src_p, 'r', encoding='utf-8') if l.strip()]
    tgts = [l.strip() for l in open(tgt_p, 'r', encoding='utf-8') if l.strip()]
    preds = predict_ct2(srcs, beam_size=1, rep_penalty=1.2)
    acc = compute_acc(preds, tgts)
    arm_g_results["suites"][s_name] = acc
    suite_accs.append(acc)
    print(f"  {s_name:<20}: Acc = {acc:6.2f}% ({len(srcs):,} pairs)")

mean_acc = sum(suite_accs) / len(suite_accs)
arm_g_results["mean_acc"] = mean_acc
print(f"\n>> OVERALL 5-SUITE MEAN ACCURACY (ARM G): {mean_acc:6.2f}%")

# Evaluate 18 Zero Click cases
zc_srcs = [c[0] for c in ZERO_CLICK_CASES]
zc_tgts = [c[1] for c in ZERO_CLICK_CASES]
zc_preds_b1 = predict_ct2(zc_srcs, beam_size=1, rep_penalty=1.2)
zc_preds_b4 = predict_ct2(zc_srcs, beam_size=4, rep_penalty=1.2)
zc_acc_b1 = compute_acc(zc_preds_b1, zc_tgts)
zc_acc_b4 = compute_acc(zc_preds_b4, zc_tgts)
arm_g_results["zero_click_acc_beam1"] = zc_acc_b1
arm_g_results["zero_click_acc_beam4"] = zc_acc_b4
arm_g_results["zero_click_preds"] = {s: p for s, p in zip(zc_srcs, zc_preds_b1)}
print(f"Zero-Click Accuracy (18 cases): Beam 1 = {zc_acc_b1:5.1f}% | Beam 4 = {zc_acc_b4:5.1f}%")


In [ ]:
# Latency profiling on CPU INT8
test_queries = [
    "dh bach khoa ha noi",
    "86 xo viet nghe tinh p19 binh thanh",
    "bv cho ray",
    "duong le van viet q9",
    "kcn song than 1"
]

# Beam 1
lats_b1 = []
for q in test_queries * 10:
    toks = tokenizer.tokenize(q)
    t0 = time.perf_counter()
    translator_cpu.translate_batch([toks], beam_size=1, repetition_penalty=1.2)
    lats_b1.append((time.perf_counter() - t0) * 1000.0)
lats_b1.sort()
p50_b1 = lats_b1[len(lats_b1) // 2]
p90_b1 = lats_b1[int(len(lats_b1) * 0.9)]
lats_b4 = []
for q in test_queries * 10:
    toks = tokenizer.tokenize(q)
    t0 = time.perf_counter()
    translator_cpu.translate_batch([toks], beam_size=4, repetition_penalty=1.2)
    lats_b4.append((time.perf_counter() - t0) * 1000.0)
lats_b4.sort()
p50_b4 = lats_b4[len(lats_b4) // 2]
p90_b4 = lats_b4[int(len(lats_b4) * 0.9)]

arm_g_results["latency"]["cpu_p50_beam1_ms"] = p50_b1
arm_g_results["latency"]["cpu_p90_beam1_ms"] = p90_b1
arm_g_results["latency"]["cpu_p50_beam4_ms"] = p50_b4
arm_g_results["latency"]["cpu_p90_beam4_ms"] = p90_b4

print(f"Latency CPU INT8 (4 threads): Beam 1 P50 = {p50_b1:.2f}ms (P90 = {p90_b1:.2f}ms) | Beam 4 P50 = {p50_b4:.2f}ms (P90 = {p90_b4:.2f}ms)")


In [ ]:
# Baseline measurements from Arms A-F for direct side-by-side comparison
BASELINES = {
    "Arm A (2E1D d128)":  {"enc_dec": "2E/1D",   "params": "6.5M",  "p50": "2.38ms",  "mean_acc": 63.50, "plasticity": 41.7, "retention": 70.2, "user_centric": 58.3, "protection_heldout": 56.8, "zc": "77.8%"},
    "Arm E (2E2D d128)":  {"enc_dec": "2E/2D",   "params": "7.1M",  "p50": "5.71ms",  "mean_acc": 67.33, "plasticity": 43.6, "retention": 73.0, "user_centric": 66.7, "protection_heldout": 62.4, "zc": "66.7%"},
    "Arm B (4E4D d128)":  {"enc_dec": "4E/4D",   "params": "9.6M",  "p50": "5.51ms",  "mean_acc": 67.11, "plasticity": 43.7, "retention": 74.2, "user_centric": 66.7, "protection_heldout": 61.0, "zc": "72.2%"},
    "Arm D (4E1D d128)":  {"enc_dec": "4E/1D",   "params": "7.7M",  "p50": "7.25ms",  "mean_acc": 65.87, "plasticity": 42.4, "retention": 71.1, "user_centric": 66.7, "protection_heldout": 58.4, "zc": "77.8%"},
    "Arm F (2E1D d256)":  {"enc_dec": "2E/1D",   "params": "13.4M", "p50": "3.97ms",  "mean_acc": 64.76, "plasticity": 42.4, "retention": 72.5, "user_centric": 58.3, "protection_heldout": 58.8, "zc": "83.3%"},
    "Arm C (6E6D d128)":  {"enc_dec": "6E/6D",   "params": "12.1M", "p50": "17.63ms", "mean_acc": 64.23, "plasticity": 42.7, "retention": 72.8, "user_centric": 58.3, "protection_heldout": 59.6, "zc": "88.9%"},
    "Arm G (ViT5-base)":  {"enc_dec": "12E/12D", "params": "226M",  "p50": "88.75ms", "mean_acc": 73.43, "plasticity": 49.2, "retention": 78.3, "user_centric": 75.0, "protection_heldout": 71.2, "zc": "66.7%"},
}

g_suites = arm_g_results["suites"]
g_p50 = f"{p50_b1:.2f}ms"
g_mean = arm_g_results["mean_acc"]
g_telex = g_suites.get("plasticity", 0)
g_osm = g_suites.get("retention", 0)
g_user = g_suites.get("user_centric", 0)
g_heldout = g_suites.get("protection_heldout", 0)
g_zc = f"{arm_g_results['zero_click_acc_beam1']:.1f}%"

lines = []
lines.append("# Báo Cáo Đối Sánh Thực Nghiệm: Pilot Arm G (ViT5-base) vs Arms A–F")
lines.append("\nSo sánh sòng phẳng giữa mô hình tiền huấn luyện tiếng Việt (ViT5-base 220M) và các kiến trúc Transformer huấn luyện từ scratch trên cùng 300.000 cặp câu sạch và 10.000 bước huấn luyện.\n")
lines.append("## 1. Bảng Đối Sánh Tổng Hợp")
lines.append("| Arm | Kiến Trúc | Params | Pretrained? | CPU P50 | Mean Acc (5 Suites) | Telex (Plasticity) | OSM (Retention) | User Centric | Protection Heldout | Zero-Click (18) |")
lines.append("| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |")

for arm_name, d in BASELINES.items():
    lines.append(f"| **{arm_name}** | {d['enc_dec']} | {d['params']} | Không (Scratch) | {d['p50']} | **{d['mean_acc']:.2f}%** | {d['telex']:.1f}% | {d['osm']:.1f}% | {d['user']:.1f}% | {d['heldout']:.1f}% | {d['zc']} |")

lines.append(f"| **Arm G (ViT5-base)** | 12E/12D | {total_params / 1e6:.1f}M | **CÓ (Pretrained)** | {g_p50} | **{g_mean:.2f}%** | {g_telex:.1f}% | {g_osm:.1f}% | {g_user:.1f}% | {g_heldout:.1f}% | {g_zc} |")

delta_vs_a = g_mean - BASELINES["Arm A (2E1D)"]["mean_acc"]
delta_vs_e = g_mean - BASELINES["Arm E (2E2D)"]["mean_acc"]

lines.append("\n---\n")
lines.append("## 2. Phân Tích Khoa Học")
lines.append(f"- **Chênh lệch so với Baseline Scratch (Arm A 2E1D)**: **{delta_vs_a:+.2f}%** ({g_mean:.2f}% vs 63.50%)")
lines.append(f"- **Chênh lệch so với Mô hình Scratch Tốt Nhất (Arm E 2E2D)**: **{delta_vs_e:+.2f}%** ({g_mean:.2f}% vs 67.33%)")
lines.append(f"- **Đánh đổi về Độ Trễ**: Arm A mất {BASELINES['Arm A (2E1D)']['p50']}, Arm E mất {BASELINES['Arm E (2E2D)']['p50']}, trong khi ViT5-base mất {g_p50} trên CPU.")

lines.append("\n---\n")
lines.append("## 3. Dự Đoán Trên 18 Ca Lỗi Thực Tế (`zero_click.csv`)")
lines.append("| Truy vấn | Nhãn mong đợi | Arm G (ViT5-base) Dự đoán | Trạng thái |")
lines.append("| :--- | :--- | :--- | :---: |")
for q, exp in ZERO_CLICK_CASES:
    p = arm_g_results["zero_click_preds"].get(q, "")
    st = "Đúng" if p.strip().lower() == exp.strip().lower() else "⚠️ Khác biệt"
    lines.append(f"| `{q}` | **{exp}** | `{p}` | {st} |")

report_text = "\n".join(lines) + "\n"
report_file = WORKING / "scaling_laws_report_arm_g.md"
results_json = WORKING / "benchmark_results_arm_g.json"

with open(report_file, 'w', encoding='utf-8') as f:
    f.write(report_text)
with open(results_json, 'w', encoding='utf-8') as f:
    json.dump(arm_g_results, f, indent=2, ensure_ascii=False)

print(report_text)
print(f"\n[+] Saved report to: {report_file}")
print(f"[+] Saved JSON results to: {results_json}")


In [ ]:
print("\n" + "=" * 80)
print("PACKAGING ARM G RESULTS FOR EASY 1-CLICK DOWNLOAD")
print("=" * 80)

archive_base = WORKING / "arm_g_vit5_results"
shutil.make_archive(
    str(archive_base),
    'zip',
    root_dir=str(OUTPUT_ROOT.parent),
    base_dir="arm_g_vit5_base"
)
zip_path = archive_base.with_suffix('.zip')
size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f"[SUCCESS] Archive created: {zip_path} ({size_mb:.1f} MB)")
print("Bạn có thể tải file này từ thanh bên phải (Output) của giao diện Kaggle về máy!")
